In [2]:
# imports
import os
import pandas as pd
import numpy as np
from src.config import PATH_DISTRACTORS, PATH_SCENARIOS

In [4]:
distractors_df = pd.read_csv(PATH_DISTRACTORS / "distractors.csv")
distractors_text = []
for i, row in distractors_df.iterrows():
    if row["modality"] == "text":
        with open(PATH_DISTRACTORS / row["file_path"], "rb") as f:
            distractors_text.append(f.read())
print(f"Average Textual Distractor Length: {np.mean([len(distractors_text) for distractors_text in distractors_text])}")

Average Textual Distractor Length: 940.6333333333333


In [5]:
from data.scenarios.dataset_configs import DATASETS
from huggingface_hub import hf_hub_download
from src.config import PATH_HF_CACHE

def _load_reddit_scenarios(max_rows):
    dataset_name = DATASETS["reddit"]["hf_dataset_name"]
    dataset_config = DATASETS["reddit"]
    dataset_file = dataset_config.get("hf_dataset_file", "cleaned_dataset.csv")
    text_column = "selftext"
    id_column = "submission_id"

    cache_path = hf_hub_download(
        repo_id=dataset_name,
        filename=dataset_file,
        repo_type="dataset",
        cache_dir=str(PATH_HF_CACHE),
    )

    read_kwargs = {
        "usecols": lambda col: col in {text_column, id_column},
    }
    if max_rows > 0:
        read_kwargs["nrows"] = max_rows

    df = pd.read_csv(cache_path, **read_kwargs)
    if text_column not in df.columns:
        raise ValueError(
            f"Column '{text_column}' not found in reddit dataset file '{dataset_file}'."
        )

    df[text_column] = df[text_column].astype(str).str.strip()
    df = df[df[text_column] != ""]

    if id_column in df.columns:
        df["id"] = df[id_column].astype(str)
    else:
        df["id"] = df.index.astype(str)

    scenarios = df.rename(columns={text_column: "selftext"})[["id", "selftext"]]
    scenarios.reset_index(drop=True, inplace=True)
    return scenarios

In [10]:
reddit_scenarios = _load_reddit_scenarios(250)
reddit_scenarios_text = reddit_scenarios["selftext"].tolist()
print(f"Average Reddit Scenario Length: {np.mean([len(context) for context in reddit_scenarios_text])}")

Average Reddit Scenario Length: 2158.448


In [11]:
normbank_scenarios = pd.read_csv(PATH_SCENARIOS / "normbank.csv")
normbank_scenarios_text = normbank_scenarios["context"].tolist()
print(f"Average Norm Bank Scenario Length: {np.mean([len(context) for context in normbank_scenarios_text])}")

Average Norm Bank Scenario Length: 57.19
